In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfterMPI\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfterMPIDerivativeByFluxexclusive\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [4]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL-Partitioning");

In [5]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL-Partitioning");

Project name is set to 'Droplet-EE-LikeEL-Partitioning'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-Partitioning'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-Partitioning'.


## Create grid

In [83]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 2;
//        double yTop = 1.5 - 20.0 / 176.7;
//        double yBottom = -20.0 / 176.7;
        double yTop = 2;
        double yBottom = -2;
        //int kelem = 8;
        int kelem = 32;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
//        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 8 * 3 + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;

                    return et;
                });
                
        
                grd.AddPredefinedPartitioning("2ProcSplit", delegate (double[] X) {
                    int rank;
                    double x = X[0];
                    
                    if(x <= 0)
                        rank = 0;
                    else
                        rank = 1;
                
                    return rank;
                });
        
                grd.AddPredefinedPartitioning("8ProcSplit", delegate (double[] X) {
                    int rank;
                    double x = X[0];
                    
                    if(x <= - 3.0 * xSize / 4.0)
                        rank = 0;
                    else if(x <= - 2.0 * xSize / 4.0)
                        rank = 1;
                    else if(x <= - 1.0 * xSize / 4.0)
                        rank = 2;
                    else if(x <= - 0.0 * xSize / 4.0)
                        rank = 3;
                    else if(x <=   1.0 * xSize / 4.0)
                        rank = 4;
                    else if(x <=   2.0 * xSize / 4.0)
                        rank = 5;
                    else if(x <=   3.0 * xSize / 4.0)
                        rank = 6; 
                    else
                        rank = 7;
                
                    return rank;
                });
        
                grd.AddPredefinedPartitioning("RoundProcSplit", delegate (double[] X) {
                    int rank;
                    double x = X[0] + xSize / 256.0;
                    double y = X[1] + xSize / 256.0;
                    
                    //if((x * x + y * y) <= 0.04)
                    if((x * x + y * y) <= (8.0 * xSize / 64.0) * (8.0 * xSize / 64.0))
                        rank = 0;
                    else if((x * x + y * y) <= 0.36)
                        rank = 1;
                    else
                        rank = 2;
                
                    return rank;
                });
        
                return grd;
     }
 
 }

In [84]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 2 * 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           //stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    if(X[1] >= (0)) {");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
           stw.WriteLine("    }");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (0.0 - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [85]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0, int MpiNum = 1) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            //int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-DT";
            C.ProjectName = "Droplet";
            //C.SessionName = "Droplet-LikeEL-AMR" + AMRlvl + "-Merged-MPI" + MpiNum + "-TestingIO";
            C.SessionName = "Droplet-LikeEL-AMR" + AMRlvl + "-Merged-MPI" + MpiNum + "-Partitioning11";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            //C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            //C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            C.AMR_startUpSweeps = AMRlvl;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = AMRlvl});
            C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[] { 0, 1 }, FieldWidth = 1 });
            //C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[], FieldWidth = 1 });
            //C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl, levelSet = 0 });

            #endregion
                
            C.GridPartType = GridPartType.Predefined;
            C.GridPartOptions = "RoundProcSplit";
            
            //C.DynamicLoadBalancing_On = true;
            //C.DynamicLoadBalancing_Period = 50;
            //C.DynamicLoadBalancing_RedistributeAtStartup = true;
            //C.GridPartType = GridPartType.METIS;


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 2e-5;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 1;
            C.saveperiod = 1;

            C.TracingNamespaces = "*";
            
            #endregion
            
            return C;
        }

## Send and run jobs

In [86]:
int[] MpiNUMs = new int[] {3};
//int[] MpiNUMs = new int[] {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12};

In [87]:
foreach(int MpiNUM in MpiNUMs){
    var C_CTRLFILE = Aland(2, 0, MpiNUM);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = MpiNUM;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput();
    }

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Droplet-LikeEL-AMR0-Merged-MPI3-Partitioning11 ... 
Set Database: { Session Count = 63; Grid Count = 130; Path = C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-Partitioning }
Grid is not in database yet...
Grid successfully saved: 75d8d51d-de73-4f09-84c0-f69e00c9d89d


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\Deployment\Droplet-EE-LikeEL-Partitioning-ZwoLevelSetSolver2024Jun14_203826.350144
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [49]:
//wmg.DefaultDatabase.Sessions

In [13]:
//wmg.DefaultDatabase.Sessions[0].Timesteps.Count

3

In [88]:
var outPath = wmg.DefaultDatabase.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-Partitioning__Droplet-LikeEL-AMR0-Merged-MPI3-Partitioning11__f99f4317-0edd-4874-aeb2-a14f550329ba


In [15]:
//for (int i = 0; i < 8; i++){
//    var outPath = wmg.DefaultDatabase.Sessions[i].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();
//}